# NB6 ? Formula-AS Feature Ablation and SHAP Diagnostics

This notebook is diagnostic, not part of the core B0-B3 training chain.

Goals:
- Compare a short formula-AS PPO training run with different no-leakage feature sets.
- Use SHAP only as policy interpretability: it explains which inputs move `[u_gamma, u_skew]`, not which inputs causally improve PnL.
- Keep the final thesis comparison clean: final B1-B3 should be trained after the feature set is chosen.


In [ ]:
import sys
import pathlib
from dataclasses import replace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
     if (p / "procs").exists()),
    pathlib.Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stable_baselines3 import PPO

from procs.gym.experiment_config import ReplayExperimentConfig
from procs.gym.data_loader import load_multi_day_with_features
from procs.gym.calibration import calibrate_from_arrays
from procs.gym.formula_as import FORMULA_FEATURE_NAMES
from procs.gym.sb3_wrapper import StableBaselinesTradingEnvironment
from procs.gym.notebook_support import (
    build_formula_multi_day_replay_env,
    build_formula_replay_env,
    evaluate_formula_rl_per_day,
    freeze_vecnorm,
    make_vecnorm,
)
from procs.rewards import CjMmCriterion

cfg = ReplayExperimentConfig()
cfg.ensure_artifact_dirs()
print(f"Repo root: {cfg.repo_root}")


## Section 1 ? Data Split and Train-Only Calibration

The ablation uses days 1-4 for short training and days 5-6 for validation. The final thesis models still use days 1-6 for training and days 7-29 for reporting.


In [ ]:
TRAIN_DAYS = 6
ABLATION_TRAIN_DAYS = 4
ABLATION_VAL_DAYS = 2

daily_S, daily_dt, dates, daily_features = load_multi_day_with_features(
    str(cfg.datasets_dir),
    pair=cfg.pair,
    tick_size=cfg.tick_size,
)

train_S = daily_S[:ABLATION_TRAIN_DAYS]
train_dt = daily_dt[:ABLATION_TRAIN_DAYS]
train_features = daily_features[:ABLATION_TRAIN_DAYS]
train_dates = dates[:ABLATION_TRAIN_DAYS]

val_S = daily_S[ABLATION_TRAIN_DAYS:TRAIN_DAYS]
val_dt = daily_dt[ABLATION_TRAIN_DAYS:TRAIN_DAYS]
val_features = daily_features[ABLATION_TRAIN_DAYS:TRAIN_DAYS]
val_dates = dates[ABLATION_TRAIN_DAYS:TRAIN_DAYS]

param_rows = []
for date, S, dt in zip(dates[:TRAIN_DAYS], daily_S[:TRAIN_DAYS], daily_dt[:TRAIN_DAYS]):
    sigma, A, kappa = calibrate_from_arrays(S, dt, tick_size=cfg.tick_size)
    param_rows.append({"Day": date, "sigma": sigma, "A": A, "kappa": kappa})

df_train_params = pd.DataFrame(param_rows).set_index("Day")
sigma_train = float(df_train_params["sigma"].median())
A_train = float(df_train_params["A"].median())
kappa_train = float(df_train_params["kappa"].median())
cfg = replace(cfg, A=A_train, kappa=kappa_train)

FORMULA_KWARGS = dict(
    sigma=sigma_train,
    gamma_min=0.01,
    gamma_max=0.9,
    skew_ticks_max=5.0,
)

print(f"Ablation train days: {train_dates}")
print(f"Validation days    : {val_dates}")
print(f"Train-only params  : sigma={sigma_train:.8f}, A={A_train:.6f}, kappa={kappa_train:.2f}")


## Section 2 ? Short Feature Ablation

Recommended interpretation:
- If `base_state` is close to `full`, the extra features are not buying much.
- If `full` improves validation drawdown/PnL, use the full feature set for final B1-B3.
- If a feature set improves training but not validation, treat it as overfitting.


In [ ]:
FEATURE_SETS = {
    "base_state": (
        "inventory_norm",
        "time_remaining_frac",
        "log_mid_rel",
    ),
    "base_plus_vol": (
        "inventory_norm",
        "time_remaining_frac",
        "log_mid_rel",
        "rolling_vol_ticks",
    ),
    "full": tuple(FORMULA_FEATURE_NAMES),
}

ABLATION_TIMESTEPS = 100_000
reward_fn = CjMmCriterion(per_step_inventory_aversion=cfg.phi)
rows = []

for name, features in FEATURE_SETS.items():
    print()
    print(f"=== Training ablation: {name} ===")
    env = build_formula_multi_day_replay_env(
        train_S,
        train_dt,
        cfg,
        daily_market_features=train_features,
        reward_fn=reward_fn,
        mode="sequential",
        enabled_features=features,
        **FORMULA_KWARGS,
    )
    sb3 = StableBaselinesTradingEnvironment(env)
    vn = make_vecnorm(sb3, cfg, training=True, norm_reward=True)

    model = PPO(
        "MlpPolicy",
        vn,
        **cfg.ppo_kwargs(),
        tensorboard_log=str(cfg.repo_root / "tb_logs" / f"formula_ablation_{name}"),
        verbose=0,
        device="cpu",
    )
    model.learn(total_timesteps=ABLATION_TIMESTEPS)

    model_stem = f"ppo_formula_ablation_{name}"
    vn_stem = f"vecnorm_formula_ablation_{name}"
    model.save(str(cfg.model_path(model_stem)))
    vn.save(str(cfg.vecnorm_path(vn_stem)))

    df_val = evaluate_formula_rl_per_day(
        model=model,
        vecnorm_path=cfg.vecnorm_path(vn_stem),
        test_S=val_S,
        test_dt=val_dt,
        test_dates=val_dates,
        test_market_features=val_features,
        config=cfg,
        seed=cfg.evaluation_seed,
        num_rollouts=cfg.evaluation_rollouts,
        enabled_features=features,
        **FORMULA_KWARGS,
    )
    df_val.to_csv(cfg.result_path(f"formula_ablation_{name}_validation.csv"))
    rows.append({
        "feature_set": name,
        "n_features": len(features),
        "features": ", ".join(features),
        "mean_sharpe": df_val["Sharpe"].mean(),
        "mean_maxdd": df_val["Max DD"].mean(),
        "mean_pnl": df_val["Final PnL"].mean(),
        "mean_abs_q": df_val["Mean |q|"].mean(),
    })

ablation_summary = pd.DataFrame(rows).set_index("feature_set")
ablation_summary.to_csv(cfg.result_path("formula_feature_ablation_summary.csv"))
print(ablation_summary.to_string(float_format="{:.5f}".format))


## Section 3 ? SHAP Policy Explanation

This explains the trained actor's deterministic action outputs, not final profitability. Use it as a descriptive statement such as: ?the policy's gamma/skew decisions are most sensitive to inventory and volatility.? Do not claim causality from SHAP alone.


In [ ]:
# Run this after final B1 has been trained with the full formula feature set.
try:
    import shap
    import torch
except ImportError as exc:
    raise ImportError("Install shap and torch to run this diagnostic cell.") from exc

model = PPO.load(str(cfg.model_path("ppo_b1_doge")), device="cpu")
features = tuple(FORMULA_FEATURE_NAMES)

env = build_formula_replay_env(
    daily_S[0],
    daily_dt[0],
    cfg,
    sigma=sigma_train,
    market_features=daily_features[0],
    reward_fn=CjMmCriterion(per_step_inventory_aversion=cfg.phi),
    enabled_features=features,
    **{k: v for k, v in FORMULA_KWARGS.items() if k != "sigma"},
)
sb3 = StableBaselinesTradingEnvironment(env)
eval_vn = freeze_vecnorm(cfg.vecnorm_path("vecnorm_b1"), sb3, cfg, norm_reward=False)

obs, _ = env.reset(seed=cfg.evaluation_seed)
obs_samples = []
for _ in range(2_000):
    obs_samples.append(obs[0].copy())
    norm_obs = eval_vn.normalize_obs(obs.copy())
    action, _ = model.predict(norm_obs[0], deterministic=True)
    obs, _, done, _, _ = env.step(action.reshape(1, -1))
    if bool(done[0]):
        break

X = np.asarray(obs_samples, dtype=np.float32)
background = X[::max(1, len(X)//100)][:100]
explain = X[::max(1, len(X)//200)][:200]

def policy_action_fn(batch):
    norm_batch = eval_vn.normalize_obs(np.asarray(batch, dtype=np.float32))
    actions = []
    for row in norm_batch:
        action, _ = model.predict(row, deterministic=True)
        actions.append(action)
    return np.asarray(actions)

explainer = shap.KernelExplainer(policy_action_fn, background)
shap_values = explainer.shap_values(explain, nsamples=100)

for output_idx, output_name in enumerate(["u_gamma", "u_skew"]):
    vals = shap_values[output_idx] if isinstance(shap_values, list) else shap_values[:, :, output_idx]
    mean_abs = np.abs(vals).mean(axis=0)
    ranking = pd.Series(mean_abs, index=features).sort_values(ascending=False)
    print()
    print(f"SHAP ranking for {output_name}:")
    print(ranking.to_string(float_format="{:.6f}".format))
    ranking.to_csv(cfg.result_path(f"shap_formula_b1_{output_name}.csv"))

shap.summary_plot(shap_values, explain, feature_names=list(features), show=True)
